# Module 1 — Part 1: RAG Pipeline

Lessons 3–7: What is RAG, dataset, search, prompt, LLM.

**Provider note:** Lessons use OpenAI `responses.create`. We use Groq `chat.completions.create` — OpenAI-compatible, free tier, fast.

## Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

from openai import OpenAI

# Groq is OpenAI-compatible — same client, different base_url
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

## Lesson 3 — What is RAG

First: call LLM directly with no context. It gives generic answers for course-specific questions.

In [ ]:
def llm(prompt, model=MODEL):
    # Lesson uses: openai_client.responses.create(model=..., input=prompt)
    # Groq uses chat.completions with messages list
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

llm("Hey, what's up?")

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)
# Generic answer — LLM has no idea about our specific Zoomcamp courses

### Manual context injection

Paste FAQ entries directly into the prompt. This is naive RAG — but it shows the concept.

In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)
# Now the answer is correct and grounded in our FAQ data — this is RAG

## Lesson 4 — The Course FAQ Dataset

Fetch all FAQ documents from DataTalks.Club. These are our knowledge base.

In [ ]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

print(f"Found {len(courses_raw)} courses")
courses_raw[:2]

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

print(f"Total documents: {len(documents)}")

In [ ]:
# Each document has: id, course, section, question, answer
documents[0]

## Lesson 5 — Search

Build a search index with minsearch. Text fields are tokenized + ranked. Keyword fields are exact-match filters.

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],  # tokenized, ranked
    keyword_fields=["course"]                         # exact filter, like SQL WHERE
)

index.fit(documents)
print("Index built")

In [ ]:
# boost_dict: question matches count 2x, section matches count 0.5x
# filter_dict: only return results from llm-zoomcamp
search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

[doc["question"] for doc in search_results]

In [ ]:
def search(question, course="llm-zoomcamp"):
    return index.search(
        question,
        boost_dict={"question": 2.0, "section": 0.5},
        filter_dict={"course": course},
        num_results=5
    )

search_results = search(question)
search_results

## Lesson 6 — Building the Prompt

Format search results into a context string, then combine with the question into a user prompt.

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    return USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    ).strip()

prompt = build_prompt(question, search_results)
print(prompt)

## Lesson 7 — The LLM + Full RAG Pipeline

Wire search + prompt + LLM together. Use message history to separate instructions from user prompt.

In [ ]:
def llm(instructions, user_prompt, model=MODEL):
    # Lesson uses responses.create with 'developer' role.
    # Groq chat.completions uses 'system' role — functionally identical.
    message_history = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = groq_client.chat.completions.create(
        model=model,
        messages=message_history
    )
    return response.choices[0].message.content

# Check token usage
raw_response = groq_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt}
    ]
)
print("Usage:", raw_response.usage)
print(raw_response.choices[0].message.content)

In [ ]:
def rag(query, model=MODEL):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

# Full RAG pipeline in one call
answer = rag("I just discovered the course. Can I join now?")
print(answer)

In [ ]:
# Try more questions
print(rag("How do I get a certificate?"))

In [ ]:
print(rag("Is it okay if I submit my homework a day late?"))